# Azure Databricks — Self-Service Connection Notebook

Query the shared Databricks warehouse from your laptop **without a Personal Access Token** and **without owning a cluster**. Sign in once with your P&G account (device code), and your session is remembered for ~1 hour.

## Prerequisites
- Python 3.9+ with `pip`
- Your P&G Azure AD account (you'll log in via a browser code — no password stored)
- Access granted to the Databricks workspace / catalog you want to query
- The target **all-purpose cluster ID** (from Databricks → **Compute** → your cluster → JDBC/ODBC tab)

## Do I need to start the cluster first? — No.
You do **not** need to open Databricks and start the cluster manually. This notebook checks the cluster state and, if it's stopped (they auto-stop after ~30 min idle), **starts it for you and waits until it's ready** — printing a heartbeat while it warms up (cold start takes a few minutes). If it's already running, queries go straight through.

## How it works
1. **Auth** — Microsoft device-code login. A token is cached to a local file so re-runs don't re-prompt; it renews silently via a cached refresh token.
2. **Connect** — uses `databricks-sql-connector` over the cluster's HTTP Path. This works with Unity Catalog **shared / USER_ISOLATION** clusters (the legacy REST 1.2 API does not).
3. **Query** — call `query("SELECT ...")` → returns a pandas DataFrame.

> **To share:** send this single `.ipynb` file. Each person edits the **Config** cell with the workspace host + cluster ID(s), then runs top to bottom. No other files needed.


## 1. Install required libraries

Run once. `databricks-sql-connector` is the official driver; `msal` handles the device-code login.


In [3]:
%pip install --quiet requests pandas msal databricks-sql-connector tabulate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Config — edit these for your workspace

Get the **hostname** and **HTTP Path** from Databricks → **Compute** → *your cluster* → **Advanced options → JDBC/ODBC**.  
The HTTP Path looks like `sql/protocolv1/o/<org_id>/<cluster_id>`. Only the two values below usually change.


In [1]:
# ───────────────────────── EDIT THIS BLOCK ─────────────────────────
SERVER_HOSTNAME = "adb-3125606391614073.13.azuredatabricks.net"   # Compute → cluster → JDBC/ODBC
ORG_ID          = "3125606391614073"                              # the "o/<org_id>" part of HTTP Path

# All-purpose clusters you can use, by friendly name → cluster id.
# NOTE: both clusters share ONE Unity Catalog metastore, so they see the SAME
# catalogs/tables. Picking a cluster only chooses which compute runs the query.
CLUSTERS = {
    "WCS": "0227-084635-3intfklw",   # odp_wcs_cluster
    "WHS": "0319-122402-5abz8tts",   # odp_wh_cluster (warehousing)
}
ACTIVE_CLUSTER = "WHS"               # key from CLUSTERS above (or paste a raw cluster id)

# Your Azure AD tenant. P&G = the value below; change only if your org differs.
TENANT_ID = "3596192b-fdf5-4e2c-a6fa-acb706c963d8"
# ────────────────────────────────────────────────────────────────────

CLUSTER_ID = CLUSTERS.get(ACTIVE_CLUSTER, ACTIVE_CLUSTER)
HTTP_PATH  = f"sql/protocolv1/o/{ORG_ID}/{CLUSTER_ID}"
BASE_URL   = f"https://{SERVER_HOSTNAME}"
print(f"Workspace : {SERVER_HOSTNAME}")
print(f"Cluster   : {ACTIVE_CLUSTER} = {CLUSTER_ID}")
print(f"HTTP Path : {HTTP_PATH}")

Workspace : adb-3125606391614073.13.azuredatabricks.net
Cluster   : WHS = 0319-122402-5abz8tts
HTTP Path : sql/protocolv1/o/3125606391614073/0319-122402-5abz8tts


## 3. Sign in + connection engine

Run this cell. **The first time**, it prints a URL and a code — open the URL, enter the code, and approve with your P&G account. After that you won't be prompted again.

**Your session is remembered like SSMS.** The token is cached in a **fixed folder** — `~/.databricks_session/` (your home directory) — *not* next to the notebook. This matters: a notebook's working directory isn't stable between runs, so the old version couldn't always find the cache and re-prompted. With a fixed path it reliably reuses the token (~1 hour) and renews **silently** via the cached refresh token for weeks. The `_databricks_session.py` helper uses the **same folder**, so one login serves both.

**Full logging — nothing fails silently.** Every step prints a timestamped line, e.g. `AUTH: reusing cached token (42 min left)`, `cluster …: RUNNING`, `...query still running (20s)`, `query OK — 110 rows`. Errors are logged with their type and message before being raised.

Defines: `get_access_token()`, `ensure_cluster_running()`, `query()`, `query_on()`, `use_cluster()`.

**Slow first query?** The shared cluster auto-stops after ~30 min idle, so the first query may trigger a **cold start (a few minutes)** — you'll see the heartbeat. The engine enforces a hard **`timeout`** (default 300s, cancels instead of hanging) and **retries** transient errors. Override per call: `query(sql, timeout=600, retries=3)`.


In [2]:
import os, json, time, warnings, threading, datetime
warnings.filterwarnings("ignore")
import requests, msal, pandas as pd
from databricks import sql as dbsql

# ─────────────────────────── logging (always visible) ───────────────────────────
# Every step prints a timestamped line. Nothing fails silently: errors are logged
# with their type + message before being raised.
def log(msg, level="INFO"):
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {level:<5} {msg}", flush=True)

# ── token cache in a FIXED location (this is why re-runs won't ask you to log in
#    again — exactly like SSMS). os.getcwd() is NOT stable in notebooks, so a
#    fixed path under your home folder is used instead. The helper script
#    (_databricks_session.py) points to the same folder, so ONE login serves both.
_CACHE_DIR  = os.path.join(os.path.expanduser("~"), ".databricks_session")
os.makedirs(_CACHE_DIR, exist_ok=True)
_TOKEN_FILE = os.path.join(_CACHE_DIR, "token.json")
_MSAL_CACHE = os.path.join(_CACHE_DIR, "msal_cache.json")
log(f"session cache dir: {_CACHE_DIR}")

# Azure Databricks first-party resource id (same for every Azure Databricks workspace)
_SCOPE     = "2ff814a6-3304-4ab8-85cb-cd0e6f879c1d/.default"
# Azure CLI public client — pre-authorised for Azure Databricks device-code login
_CLIENT_ID = "04b07795-8ddb-461a-bbee-02f9e1bf7b46"
_AUTHORITY = f"https://login.microsoftonline.com/{TENANT_ID}"

# ── tuning knobs (safe defaults) ──
QUERY_TIMEOUT   = 300   # seconds to wait for a single query before cancelling
QUERY_RETRIES   = 2     # extra attempts on transient/connection errors
CLUSTER_TIMEOUT = 600   # seconds to wait for a cold cluster to reach RUNNING

_conn = {"cluster": None, "handle": None}


def get_access_token():
    """Return a valid AAD token. Logs exactly where the token came from so you can
    see WHY (or why not) a login is required. Order: file cache -> silent -> device."""
    # 1) raw token file (fast path — no MSAL work at all)
    if os.path.exists(_TOKEN_FILE):
        try:
            d = json.load(open(_TOKEN_FILE))
            remaining = int((d.get("expires_at", 0) - time.time()) / 60)
            if time.time() < d.get("expires_at", 0) - 300:
                log(f"AUTH: reusing cached token ({remaining} min left) — no login needed")
                return d["access_token"]
            log("AUTH: cached token expired — trying silent refresh")
        except Exception as e:
            log(f"AUTH: token file unreadable ({e}) — will re-auth", "WARN")
    else:
        log("AUTH: no cached token yet (first run on this machine)")

    # 2) MSAL silent (uses refresh token) / 3) device code (interactive)
    cache = msal.SerializableTokenCache()
    if os.path.exists(_MSAL_CACHE):
        cache.deserialize(open(_MSAL_CACHE).read())
    app = msal.PublicClientApplication(_CLIENT_ID, authority=_AUTHORITY, token_cache=cache)
    result, accts = None, app.get_accounts()
    log(f"AUTH: {len(accts)} cached account(s) available for silent renewal")
    for acct in accts:
        result = app.acquire_token_silent([_SCOPE], account=acct)
        if result:
            log(f"AUTH: silent renewal succeeded for {acct.get('username')}")
            break
    if not result:
        log("AUTH: silent renewal not possible — starting device-code login", "WARN")
        flow = app.initiate_device_flow(scopes=[_SCOPE])
        if "user_code" not in flow:
            raise RuntimeError(f"device flow init failed: {flow}")
        print("\n" + "=" * 72)
        print(">>> " + flow["message"])          # open the URL and enter the code
        print("=" * 72 + "\n", flush=True)
        result = app.acquire_token_by_device_flow(flow)
    if "access_token" not in result:
        raise RuntimeError(result.get("error_description", "Login failed"))
    if cache.has_state_changed:
        open(_MSAL_CACHE, "w").write(cache.serialize())
        log("AUTH: refresh token saved — future runs will renew silently")
    json.dump({"access_token": result["access_token"],
               "expires_at": time.time() + result.get("expires_in", 3600),
               "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")},
              open(_TOKEN_FILE, "w"))
    log("AUTH: new token cached")
    return result["access_token"]


def _cluster_state():
    r = requests.get(f"{BASE_URL}/api/2.0/clusters/get",
                     headers={"Authorization": f"Bearer {get_access_token()}"},
                     params={"cluster_id": CLUSTER_ID}, timeout=60)
    r.raise_for_status()
    d = r.json()
    return d.get("state"), d.get("state_message", "")


def ensure_cluster_running(wait=True, poll_timeout=CLUSTER_TIMEOUT):
    """Start the cluster if stopped, and (optionally) wait until RUNNING.
    Prints a heartbeat every 15s so you can see it is working, not stuck."""
    state, msg = _cluster_state()
    log(f"cluster {CLUSTER_ID}: {state} {('- ' + msg) if msg else ''}")
    if state == "RUNNING":
        return True
    if state in ("TERMINATED", "ERROR"):
        log("cluster is stopped — starting it (cold start can take a few minutes)")
        requests.post(f"{BASE_URL}/api/2.0/clusters/start",
                      headers={"Authorization": f"Bearer {get_access_token()}"},
                      json={"cluster_id": CLUSTER_ID}, timeout=60)
    if not wait:
        return False
    t0, deadline = time.time(), time.time() + poll_timeout
    while time.time() < deadline:
        time.sleep(15)
        state, msg = _cluster_state()
        log(f"...{state}  ({int(time.time()-t0)}s elapsed)")
        if state == "RUNNING":
            log("cluster is RUNNING")
            return True
        if state in ("TERMINATED", "ERROR"):
            log(f"cluster failed to start: {msg}", "ERROR")
            return False
    log(f"timed out after {poll_timeout}s waiting for cluster", "ERROR")
    return False


def _new_connection():
    ensure_cluster_running()
    log(f"opening SQL connection to {CLUSTER_ID}")
    return dbsql.connect(server_hostname=SERVER_HOSTNAME,
                         http_path=HTTP_PATH,
                         access_token=get_access_token(),
                         _socket_timeout=QUERY_TIMEOUT)


def _connection(force_new=False):
    """Reuse a cached SQL connection; recreate if missing/stale/forced."""
    h = _conn["handle"]
    if not force_new and h is not None and _conn["cluster"] == CLUSTER_ID:
        try:
            with h.cursor() as c:
                c.execute("SELECT 1"); c.fetchall()
            return h
        except Exception:
            log("cached connection was stale — rebuilding", "WARN")
    if h is not None:
        try: h.close()
        except Exception: pass
        _conn["handle"] = None
    h = _new_connection()
    _conn.update(handle=h, cluster=CLUSTER_ID)
    return h


def _run_once(sql, timeout, force_new):
    """Run one query attempt in a worker thread so we can enforce a hard timeout
    and print a heartbeat instead of hanging silently."""
    result, error, cur_ref = {}, {}, {}

    def worker():
        try:
            cur = _connection(force_new=force_new).cursor()
            cur_ref["cur"] = cur
            cur.execute(sql)
            if cur.description is None:
                result["df"] = pd.DataFrame()
            else:
                cols = [d[0] for d in cur.description]
                result["df"] = pd.DataFrame([tuple(r) for r in cur.fetchall()], columns=cols)
            cur.close()
        except Exception as e:                       # noqa
            error["e"] = e

    th = threading.Thread(target=worker, daemon=True)
    th.start()
    t0 = time.time()
    while th.is_alive():
        th.join(timeout=10)
        if th.is_alive():
            el = int(time.time() - t0)
            if el >= timeout:
                try: cur_ref.get("cur") and cur_ref["cur"].cancel()
                except Exception: pass
                raise TimeoutError(f"query exceeded {timeout}s (cancelled)")
            log(f"...query still running ({el}s)")
    if "e" in error:
        raise error["e"]
    return result["df"]


def query(sql, timeout=QUERY_TIMEOUT, retries=QUERY_RETRIES):
    """Run SQL on the ACTIVE cluster and return a pandas DataFrame.

    - Hard `timeout` (seconds): cancels and raises instead of hanging forever.
    - Auto-`retries` on transient/connection errors, rebuilding the connection.
    - Logs every attempt + a heartbeat, so it never fails silently.
    """
    log(f"query on {CLUSTER_ID}: {' '.join(sql.split())[:90]}")
    last = None
    for attempt in range(retries + 1):
        try:
            if attempt:
                log(f"retry {attempt}/{retries}...", "WARN")
            df = _run_once(sql, timeout, force_new=bool(attempt))
            log(f"query OK — {len(df)} row(s) x {len(df.columns)} col(s)")
            return df
        except TimeoutError as e:
            last = e
            log(str(e), "ERROR")
        except Exception as e:                        # noqa
            last = e
            emsg = str(e)
            if "not authorized" in emsg or "Firewalls and virtual networks" in emsg:
                log("storage firewall blocks this table's files — the cluster's "
                    "network can't read its ADLS storage. This is a data-access "
                    "config issue, NOT your login. Try another table/cluster.", "ERROR")
                raise               # not retryable — fail fast with the clear reason
            log(f"{type(e).__name__}: {emsg[:200]}", "ERROR")
            time.sleep(min(5 * (attempt + 1), 15))    # small backoff before retry
    log("all attempts failed — raising", "ERROR")
    raise last


def use_cluster(which):
    """Switch active cluster (a key in CLUSTERS or a raw cluster id)."""
    global CLUSTER_ID, HTTP_PATH
    CLUSTER_ID = CLUSTERS.get(which, which)
    HTTP_PATH  = f"sql/protocolv1/o/{ORG_ID}/{CLUSTER_ID}"
    if _conn["handle"]:
        try: _conn["handle"].close()
        except Exception: pass
        _conn["handle"] = None
    log(f"active cluster -> {CLUSTER_ID}")


def query_on(cluster, sql, **kw):
    """Run `sql` on a SPECIFIC cluster (key in CLUSTERS or raw id), then restore
    the previously-active cluster. Lets you query schema/data from BOTH clusters."""
    prev = CLUSTER_ID
    use_cluster(cluster)
    try:
        return query(sql, **kw)
    finally:
        if prev != CLUSTER_ID:
            use_cluster(prev)


def row_counts(schema, catalog="cdl_oss_prod"):
    """Return a DataFrame of every table in `catalog.schema` with its row count,
    sorted biggest first. Empty tables show 0; unreadable tables show -1 with the
    reason logged. Use this to find which tables actually hold data."""
    tbls = query(f"SHOW TABLES IN {catalog}.{schema}")
    tcol = "tableName" if "tableName" in tbls.columns else tbls.columns[1]
    out = []
    for t in tbls[tcol]:
        fq = f"{catalog}.{schema}.{t}"
        try:
            n = int(query(f"SELECT COUNT(*) AS n FROM {fq}").iloc[0, 0])
        except Exception as e:
            log(f"{t}: cannot count ({str(e)[:80]})", "WARN")
            n = -1
        out.append({"table": t, "rows": n})
    df = pd.DataFrame(out).sort_values("rows", ascending=False).reset_index(drop=True)
    return df


def tables_with_data(schema, catalog="cdl_oss_prod"):
    """Like row_counts() but only the tables that have at least one row."""
    df = row_counts(schema, catalog)
    return df[df["rows"] > 0].reset_index(drop=True)


def preview(table, n=10):
    """Quick look: row count + first n rows for a fully-qualified table name."""
    total = int(query(f"SELECT COUNT(*) FROM {table}").iloc[0, 0])
    log(f"{table} has {total:,} rows")
    if total == 0:
        log("table is EMPTY — nothing to preview", "WARN")
        return pd.DataFrame()
    return query(f"SELECT * FROM {table} LIMIT {n}")


# warm up auth now (device code only if genuinely needed)
get_access_token()
log("✅ Signed in. Ready — use: df = query('SELECT ...')  |  query_on('WCS', sql)")
log("helpers: row_counts(schema) | tables_with_data(schema) | preview('cat.sch.tbl')")
log(f"defaults: timeout={QUERY_TIMEOUT}s, retries={QUERY_RETRIES}, cluster wait={CLUSTER_TIMEOUT}s")

[11:23:29] INFO  session cache dir: C:\Users\feng.xi.2\.databricks_session
[11:23:29] INFO  AUTH: reusing cached token (58 min left) — no login needed
[11:23:29] INFO  ✅ Signed in. Ready — use: df = query('SELECT ...')  |  query_on('WCS', sql)
[11:23:29] INFO  helpers: row_counts(schema) | tables_with_data(schema) | preview('cat.sch.tbl')
[11:23:29] INFO  defaults: timeout=300s, retries=2, cluster wait=600s


## 4. Test the connection — list catalogs

If the cluster is stopped, this will start it (takes a few minutes the first time).


In [6]:
# List catalogs you have access to
query("SHOW CATALOGS")

    cluster 0319-122402-5abz8tts: RUNNING 


[WARN] pyarrow is not installed by default since databricks-sql-connector 4.0.0,any arrow specific api (e.g. fetchmany_arrow) and cloud fetch will be disabled.If you need these features, please run pip install pyarrow or pip install databricks-sql-connector[pyarrow] to install
Token exchange failed, using external token: 'access_token'


,catalog
0,aif_use_cases_dev
1,aif_use_cases_qa
2,app_dbkp_prod
3,cdl_ama_prod
4,cdl_amhq_dev
...,...
105,lkf_timbr_virtualization_prod
106,lkf_timbr_virtualization_sl_dev
107,samples
108,system


## 5. Explore a schema & query a table

Example uses the OSS warehousing catalog `cdl_oss_prod`. Change the catalog/schema/table to whatever you have access to.


In [ ]:
# Column info for a table
df_cols = query("""
    SELECT column_name, data_type
    FROM cdl_oss_prod.information_schema.columns
    WHERE table_schema = 'silver_prime_albany'
      AND table_name   = 'dlytrn_daily_transactions'
    ORDER BY ordinal_position
""")
print(f"{len(df_cols)} columns")
df_cols

97 columns


,column_name,data_type
0,id,LONG
1,action,STRING
2,batch,STRING
3,batch_exp_date,DATE
4,block_class_minus1,STRING
...,...,...
92,shipment_qty,DECIMAL
93,goods_receipt_qty,DECIMAL
94,goods_receipt_date,DATE
95,in_npi_counter_days,INT


In [ ]:
# Query actual data — this table has ~18 million rows, so LIMIT keeps it fast.
df = query("""
    SELECT *
    FROM cdl_oss_prod.silver_prime_albany.dlytrn_daily_transactions
    LIMIT 100
""")
print(f"{len(df)} rows × {len(df.columns)} columns")
df.head(10)

NameError: name 'query' is not defined

## 5b. Find tables that actually have data

If a query returns **0 rows**, the table is simply empty — not an error. Some tables in this warehouse are empty placeholders, and a few are blocked by a storage firewall (you'll get a clear message, not a silent fail). Use these helpers to see what's populated before you query.

- `row_counts("schema")` — every table + its row count (biggest first)
- `tables_with_data("schema")` — only the non-empty tables
- `preview("catalog.schema.table")` — row count + first rows, with an empty-table warning


In [ ]:
# Which tables in this schema hold data? (biggest first)
row_counts("silver_prime_albany")


In [3]:
# Preview any populated table quickly (row count + first rows, empty-table warning)
preview("cdl_oss_prod.silver_prime_albany.invdtl_inventory_detail", n=10)


[11:16:13] INFO  query on 0319-122402-5abz8tts: SELECT COUNT(*) FROM cdl_oss_prod.silver_prime_albany.invdtl_inventory_detail
[11:16:13] INFO  AUTH: reusing cached token (65 min left) — no login needed
[11:16:14] INFO  cluster 0319-122402-5abz8tts: RUNNING 
[11:16:14] INFO  opening SQL connection to 0319-122402-5abz8tts
[11:16:14] INFO  AUTH: reusing cached token (65 min left) — no login needed


[WARN] pyarrow is not installed by default since databricks-sql-connector 4.0.0,any arrow specific api (e.g. fetchmany_arrow) and cloud fetch will be disabled.If you need these features, please run pip install pyarrow or pip install databricks-sql-connector[pyarrow] to install
Token exchange failed, using external token: 'access_token'


[11:16:17] INFO  query OK — 1 row(s) x 1 col(s)
[11:16:17] INFO  cdl_oss_prod.silver_prime_albany.invdtl_inventory_detail has 1,800,730 rows
[11:16:17] INFO  query on 0319-122402-5abz8tts: SELECT * FROM cdl_oss_prod.silver_prime_albany.invdtl_inventory_detail LIMIT 10
[11:16:21] INFO  query OK — 10 row(s) x 88 col(s)


,dtlnum,subnum,phdflg,prtnum,prt_client_id,orgcod,revlvl,lotnum,sup_lotnum,invsts,...,vc_condition_end,rtd_src_cap_dt,rtd_src_ope_st,rtd_str_int_dt,rtd_str_int_sq,rtd_str_int_sqt,kafka_timestamp,kafka_partition,kafka_offset,silver_utc_timestamp
0,D0000000ZNUH,S00000010ZWO,0,80353315,----,----,----,1225U011P,None,U,...,None,2025-11-08 07:42:17,1,2025-11-08 07:42:17.966,*|0000000009EF,*|0000000009EF,2025-11-08 07:42:21.172,0,37057,2025-11-12 09:26:17.282
1,D0000001634E,S0000001DD70,0,80335411,----,----,----,2197U011P,None,U,...,None,2025-11-08 07:42:17,1,2025-11-08 07:42:17.966,*|000000006C4E,*|000000006C4E,2025-11-08 07:42:52.027,0,62240,2025-11-12 09:26:17.282
2,D00000017PXR,S0000001GICS,0,80369914,----,----,----,2209U0184I,None,U,...,None,2025-11-08 07:42:17,1,2025-11-08 07:42:17.966,*|00000000CF54,*|00000000CF54,2025-11-08 07:43:23.546,0,87590,2025-11-12 09:26:17.282
3,D000000181VG,S0000001H30E,0,80337307,----,----,----,2209U020,None,U,...,None,2025-11-08 07:42:17,1,2025-11-08 07:42:17.966,*|000000005CCF,*|000000005CCF,2025-11-08 07:42:46.961,0,58273,2025-11-12 09:26:17.282
4,D00000018D09,S0000001HJA0,0,80343331,----,----,----,2214U018K8,None,U,...,None,2025-11-08 07:42:17,1,2025-11-08 07:42:17.966,*|00000000DB62,*|00000000DB62,2025-11-08 07:43:28.017,0,90676,2025-11-12 09:26:17.282
5,D00000018OWM,S0000001HYHX,0,80362915,----,----,----,2216U0183I,None,U,...,None,2025-11-08 07:42:17,1,2025-11-08 07:42:17.966,*|00000000AC67,*|00000000AC67,2025-11-08 07:43:11.971,0,78649,2025-11-12 09:26:17.282
6,D00000018PN2,S0000001HZF4,0,80370672,----,----,----,2216U01810,None,U,...,None,2025-11-08 07:42:17,1,2025-11-08 07:42:17.966,*|00000000DD62,*|00000000DD62,2025-11-08 07:43:28.779,0,91188,2025-11-12 09:26:17.282
7,D00000018R2J,S0000001I11W,0,80369901,----,----,----,2216U0184I,None,U,...,None,2025-11-08 07:42:17,1,2025-11-08 07:42:17.966,*|000000003662,*|000000003662,2025-11-08 07:42:35.652,0,48436,2025-11-12 09:26:17.282
8,D00000018UN6,S0000001I5II,0,80370188,----,----,----,2217U01813,None,U,...,None,2025-11-08 07:42:17,1,2025-11-08 07:42:17.966,*|000000004F13,*|000000004F13,2025-11-08 07:42:43.077,0,54757,2025-11-12 09:26:17.282
9,D0000002333S,S00000030FSA,0,91081457,----,----,----,U018338886,SCOSE129024801,U,...,None,2025-11-08 07:42:17,1,2025-11-08 07:42:17.966,*|00000000E4C6,*|00000000E4C6,2025-11-08 07:43:30.897,0,93080,2025-11-12 09:26:17.282


### Biggest populated tables (verified 2026-07-16)

Confirmed with data — good starting points:

| Table | Rows |
|---|---|
| cdl_oss_prod.silver_rtcis_capegirardeu.sysdtl_system_detail | 91,875,664 |
| cdl_oss_prod.silver_rtcis_capegirardeu.systrn_system_transactions | 59,998,460 |
| cdl_oss_prod.silver_rtcis_tabler.sysdtl_system_detail | 40,996,121 |
| cdl_oss_prod.silver_rtcis_greenbay.sysdtl_system_detail | 34,497,085 |
| cdl_oss_prod.silver_prime_albany.dlytrn_daily_transactions | 18,036,227 |
| cdl_oss_prod.gold_stt.stt_investments_customer_lvl | 3,120,794 |
| cdl_oss_prod.silver_prime_albany.dscmst_description_master | 1,912,013 |
| cdl_oss_prod.silver_prime_albany.invdtl_inventory_detail | 1,800,730 |
| cdl_oss_prod.gold_stt.stt_compliance | 918,500 |
| cdl_oss_prod.silver_fleet_control_tower.replica_shipment_status_history | 856,190 |
| cdl_oss_prod.gold_ccdl.customer_hierarchy_flattened | 162,376 |

**Known empty / blocked:** `gold_ccdl.market_operating_strategy` (0 rows), `bronze_warehousing.test_delta_table` (storage firewall — clear error, not a hang). 63 of 65 warehousing tables have data.


## 6. Browse available schemas & tables

Use these to discover what you can query in any catalog.


In [3]:
# List schemas in a catalog
catalog = "cdl_oss_prod"   # change this
query(f"SHOW SCHEMAS IN {catalog}")

NameError: name 'query' is not defined

In [4]:
# List tables in a schema
schema = "cdl_oss_prod.silver_prime_albany"   # change this
query(f"SHOW TABLES IN {schema}")

NameError: name 'query' is not defined

## 7. Save results locally

Save a DataFrame for offline analysis.


In [ ]:
# Save as CSV
# df.to_csv("power_bi_full_fct.csv", index=False)

# Save as Parquet (smaller, faster)
# df.to_parquet("power_bi_full_fct.parquet", index=False)

# To reload later:
# df = pd.read_parquet("power_bi_full_fct.parquet")

## 8. Use both clusters — WCS & WHS

Two clusters are pre-registered: **WCS** (`odp_wcs_cluster`) and **WHS** (`odp_wh_cluster`). **Both share the same Unity Catalog metastore**, so they see the **identical** catalogs and tables — verified: 110/110 catalogs match. Choosing a cluster only decides which *compute* runs your query, not what data you can reach. Use `query_on("WCS", sql)` for a one-off, or `use_cluster("WCS")` to switch the default.


In [3]:
# Run a SCHEMA query and a DATA query on BOTH clusters (auto-starts each if stopped).
# Both clusters share one metastore, so results are identical — this just proves
# each cluster's compute works and you can switch freely.
for ck in ["WCS", "WHS"]:
    log(f"================  {ck}  ================")
    # schema query
    schemas = query_on(ck, "SHOW SCHEMAS IN cdl_oss_prod")
    log(f"{ck}: {len(schemas)} schemas visible in cdl_oss_prod")
    # data query
    sample = query_on(
        ck,
        "SELECT * FROM cdl_oss_prod.silver_prime_albany.dlytrn_daily_transactions LIMIT 5",
    )
    log(f"{ck}: pulled {len(sample)} sample rows")
    display(sample)


[11:23:34] INFO  ================  WCS  ================
[11:23:34] INFO  active cluster -> 0227-084635-3intfklw
[11:23:34] INFO  query on 0227-084635-3intfklw: SHOW SCHEMAS IN cdl_oss_prod
[11:23:34] INFO  AUTH: reusing cached token (58 min left) — no login needed
[11:23:36] INFO  cluster 0227-084635-3intfklw: RUNNING 
[11:23:36] INFO  opening SQL connection to 0227-084635-3intfklw
[11:23:36] INFO  AUTH: reusing cached token (57 min left) — no login needed


[WARN] pyarrow is not installed by default since databricks-sql-connector 4.0.0,any arrow specific api (e.g. fetchmany_arrow) and cloud fetch will be disabled.If you need these features, please run pip install pyarrow or pip install databricks-sql-connector[pyarrow] to install
Token exchange failed, using external token: 'access_token'


[11:23:39] INFO  query OK — 30 row(s) x 1 col(s)
[11:23:39] INFO  active cluster -> 0319-122402-5abz8tts
[11:23:39] INFO  WCS: 30 schemas visible in cdl_oss_prod
[11:23:39] INFO  active cluster -> 0227-084635-3intfklw
[11:23:39] INFO  query on 0227-084635-3intfklw: SELECT * FROM cdl_oss_prod.silver_prime_albany.dlytrn_daily_transactions LIMIT 5
[11:23:39] INFO  AUTH: reusing cached token (57 min left) — no login needed
[11:23:40] INFO  cluster 0227-084635-3intfklw: RUNNING 
[11:23:40] INFO  opening SQL connection to 0227-084635-3intfklw
[11:23:40] INFO  AUTH: reusing cached token (57 min left) — no login needed


Token exchange failed, using external token: 'access_token'


[11:23:49] INFO  ...query still running (10s)
[11:23:59] INFO  ...query still running (20s)
[11:24:02] INFO  query OK — 5 row(s) x 109 col(s)
[11:24:02] INFO  active cluster -> 0319-122402-5abz8tts
[11:24:02] INFO  WCS: pulled 5 sample rows


,dlytrn_id,oprcod,actcod,lodnum,subnum,dtlnum,to_lodnum,to_subnum,to_dtlnum,prtnum,...,last_upd_dt,rtd_src_cap_dt,rtd_src_ope_st,rtd_str_int_dt,rtd_str_int_sq,rtd_str_int_sqt,kafka_timestamp,kafka_partition,kafka_offset,silver_utc_timestamp
0,00001OTSZP,ORDENG,ENG_EXEMP_CHG,None,None,None,None,None,None,80874931,...,2026-02-23 20:13:27,2026-03-26 06:00:06,0,2026-03-26 06:00:08.245,00000000007998766284:00001|7FFFFFFFFFFF,00000000007998766284:00001|7FFFFFFFFFFF,2026-03-26 06:00:08.269,0,37805049,2026-03-26 06:09:21.887
1,00001OTTS6,None,GENMOV,00100370009113488951,S0000008MS2U,None,None,None,None,80798314,...,2026-02-23 20:35:21,2026-03-26 06:00:28,0,2026-03-26 06:00:30.262,00000000007998769984:00001|7FFFFFFFFFFF,00000000007998769984:00001|7FFFFFFFFFFF,2026-03-26 06:00:30.287,0,37808124,2026-03-26 06:09:21.887
2,00001OTU1Y,None,IDNTFY_WO,PERM-CRE-LOD-AY,PERM-CRE-SUB-AY,D0000006W6A1,00200370009131741769,S0000008MS3Z,None,80799721,...,2026-02-23 20:44:11,2026-03-26 06:00:36,0,2026-03-26 06:00:37.368,00000000007998771316:00001|7FFFFFFFFFFF,00000000007998771316:00001|7FFFFFFFFFFF,2026-03-26 06:00:37.410,0,37809193,2026-03-26 06:09:21.887
3,00001OTUL8,ORDENG,ENG_EXEMP_CHG,None,None,None,None,None,None,80807834,...,2026-02-23 20:53:44,2026-03-26 06:00:49,0,2026-03-26 06:00:51.275,00000000007998773791:00001|7FFFFFFFFFFF,00000000007998773791:00001|7FFFFFFFFFFF,2026-03-26 06:00:51.310,0,37811284,2026-03-26 06:09:21.887
4,00001OTUQQ,None,GENMOV,00600370006309081168,S0000008LSG1,None,None,None,None,80833603,...,2026-02-23 20:56:20,2026-03-26 06:00:53,0,2026-03-26 06:00:54.158,00000000007998774604:00001|7FFFFFFFFFFF,00000000007998774604:00001|7FFFFFFFFFFF,2026-03-26 06:00:54.320,0,37811875,2026-03-26 06:09:21.887


... (2 outputs left out)

[11:24:06] INFO  query OK — 30 row(s) x 1 col(s)
[11:24:06] INFO  WHS: 30 schemas visible in cdl_oss_prod
[11:24:06] INFO  active cluster -> 0319-122402-5abz8tts
[11:24:06] INFO  query on 0319-122402-5abz8tts: SELECT * FROM cdl_oss_prod.silver_prime_albany.dlytrn_daily_transactions LIMIT 5
[11:24:06] INFO  AUTH: reusing cached token (57 min left) — no login needed
[11:24:07] INFO  cluster 0319-122402-5abz8tts: RUNNING 
[11:24:07] INFO  opening SQL connection to 0319-122402-5abz8tts
[11:24:07] INFO  AUTH: reusing cached token (57 min left) — no login needed


Token exchange failed, using external token: 'access_token'


[11:24:12] INFO  query OK — 5 row(s) x 109 col(s)
[11:24:12] INFO  WHS: pulled 5 sample rows


,dlytrn_id,oprcod,actcod,lodnum,subnum,dtlnum,to_lodnum,to_subnum,to_dtlnum,prtnum,...,last_upd_dt,rtd_src_cap_dt,rtd_src_ope_st,rtd_str_int_dt,rtd_str_int_sq,rtd_str_int_sqt,kafka_timestamp,kafka_partition,kafka_offset,silver_utc_timestamp
0,00001OTSZP,ORDENG,ENG_EXEMP_CHG,None,None,None,None,None,None,80874931,...,2026-02-23 20:13:27,2026-03-26 06:00:06,0,2026-03-26 06:00:08.245,00000000007998766284:00001|7FFFFFFFFFFF,00000000007998766284:00001|7FFFFFFFFFFF,2026-03-26 06:00:08.269,0,37805049,2026-03-26 06:09:21.887
1,00001OTTS6,None,GENMOV,00100370009113488951,S0000008MS2U,None,None,None,None,80798314,...,2026-02-23 20:35:21,2026-03-26 06:00:28,0,2026-03-26 06:00:30.262,00000000007998769984:00001|7FFFFFFFFFFF,00000000007998769984:00001|7FFFFFFFFFFF,2026-03-26 06:00:30.287,0,37808124,2026-03-26 06:09:21.887
2,00001OTU1Y,None,IDNTFY_WO,PERM-CRE-LOD-AY,PERM-CRE-SUB-AY,D0000006W6A1,00200370009131741769,S0000008MS3Z,None,80799721,...,2026-02-23 20:44:11,2026-03-26 06:00:36,0,2026-03-26 06:00:37.368,00000000007998771316:00001|7FFFFFFFFFFF,00000000007998771316:00001|7FFFFFFFFFFF,2026-03-26 06:00:37.410,0,37809193,2026-03-26 06:09:21.887
3,00001OTUL8,ORDENG,ENG_EXEMP_CHG,None,None,None,None,None,None,80807834,...,2026-02-23 20:53:44,2026-03-26 06:00:49,0,2026-03-26 06:00:51.275,00000000007998773791:00001|7FFFFFFFFFFF,00000000007998773791:00001|7FFFFFFFFFFF,2026-03-26 06:00:51.310,0,37811284,2026-03-26 06:09:21.887
4,00001OTUQQ,None,GENMOV,00600370006309081168,S0000008LSG1,None,None,None,None,80833603,...,2026-02-23 20:56:20,2026-03-26 06:00:53,0,2026-03-26 06:00:54.158,00000000007998774604:00001|7FFFFFFFFFFF,00000000007998774604:00001|7FFFFFFFFFFF,2026-03-26 06:00:54.320,0,37811875,2026-03-26 06:09:21.887
